In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import pickle

from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, make_scorer

In [2]:
# 前処理データの読み込み
train = pd.read_csv('train_balanced_pre.csv')

In [3]:
# 目的変数 '賃料' と説明変数を指定
X = train.drop(columns=['賃料'])  # 説明変数
y = train['賃料']  # 目的変数

In [4]:
# XGBoostモデルの初期化
xgb_model = xgb.XGBRegressor(
    objective = 'reg:squarederror', 
    random_state = 42, 
    n_estimators = 50,
    max_depth = 10,
    learning_rate = 0.1,
    subsample = 0.6,
    colsample_bytree = 0.8,
    n_jobs=-1
)

In [5]:
# RMSEスコアのカスタムスコア関数
def rmse_scorer(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [6]:
# クロスバリデーションでの評価 (5分割)
scores = cross_val_score(xgb_model, X, y, cv=5, scoring=make_scorer(rmse_scorer, greater_is_better=False))

In [7]:
# 各分割のRMSEを表示
print("Cross-validation RMSE scores:", -scores)

Cross-validation RMSE scores: [12634.2395529   7572.57443233  6865.17718653 28816.59498029
 81877.23343141]


In [8]:
# 平均RMSEを計算
mean_rmse = -np.mean(scores)
print(f"Mean RMSE: {mean_rmse}")

Mean RMSE: 27553.163916690995


In [9]:
# 全体データでモデルを学習
xgb_model.fit(X, y)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=50, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

In [10]:
#モデルの保存
with open("model.pickle", mode="wb") as fp:
    pickle.dump(xgb_model, fp)